# 02 — Classificador de recortes: MILCO × NOMBO

**Pergunta:** o ConvNeXt, que venceu na imagem inteira (fase 1), também vence no recorte? Contra que referência?

**Referência antiga:** a fusão GhostNetV2 + MobileNetV2 do `TESTE_JUNCAO`, reavaliada aqui nos mesmos 109 recortes verdadeiros do teste: **acurácia 0,807**. O notebook antigo reportava 0,853; a diferença provavelmente vem de outro checkpoint ou outra execução.

**Grupos testados** (todos com recortes verdadeiros do teste, margem de 15%, 224 px, AdamW + cosine, salvo indicação):

| Grupo | O que varia |
|---|---|
| B1 | Backbone e resolução |
| B2 | Otimizador e loss |
| B4 | Fusão de 2 backbones (como a fusão antiga) |
| B5 | 3 classes: MILCO, NOMBO e **fundo** (recortes aleatórios sem objeto) |
| F | Os 5 melhores com 2 sementes extras (43, 44) |

### Protocolo comum
- **Dados:** SIMD (Marinha Portuguesa, 1170 imagens side-scan, 2010–2021). **MILCO** = objeto com aparência de mina; **NOMBO** = objeto que não é mina.
- **Split:** idêntico ao notebook `sonar_deteccao_comparacao_modelos_V3_AA.ipynb` (arquivos ordenados por nome, `random.Random(42).shuffle`, 70/15/15) → 819 / 175 / 176 imagens. No teste há **109 objetos** (75 MILCO, 34 NOMBO).
- **Seleção sempre na validação** (checkpoint, conf do detector, limiar de aceite, finalista). O teste é usado uma vez, para reportar.
- **Hardware:** máquina local com 2 núcleos de CPU e 4 GB de RAM, **sem GPU**; treinos fatiados e retomáveis (`crop_harness.py` + `fila_recortes.json`).
- **Pesos antigos** (YOLO Modelo A e fusão GhostNetV2 + MobileNetV2) foram apenas lidos, nunca alterados.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3); pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 90)
sys.path.insert(0, str(Path.cwd()))
import analise_recortes as A

In [2]:
print(open("resultados/linha_base/fusao_antiga_recortes_gt.json").read())

{
 "val": {
  "acc": 0.7866666666666666,
  "f1_macro": 0.76,
  "matriz": [
   [
    42,
    8
   ],
   [
    8,
    17
   ]
  ],
  "f1_MILCO": 0.84,
  "prec_MILCO": 0.84,
  "rec_MILCO": 0.84,
  "f1_NOMBO": 0.68,
  "prec_NOMBO": 0.68,
  "rec_NOMBO": 0.68,
  "auc_MILCO_vs_NOMBO": 0.8623999999999999,
  "acc_objetos_2cls": 0.7866666666666666,
  "f1_MILCO_objetos_2cls": 0.84
 },
 "test": {
  "acc": 0.8073394495412844,
  "f1_macro": 0.7806841046277666,
  "matriz": [
   [
    63,
    12
   ],
   [
    9,
    25
   ]
  ],
  "f1_MILCO": 0.8571428571428571,
  "prec_MILCO": 0.875,
  "rec_MILCO": 0.84,
  "f1_NOMBO": 0.704225352112676,
  "prec_NOMBO": 0.6756756756756757,
  "rec_NOMBO": 0.7352941176470589,
  "auc_MILCO_vs_NOMBO": 0.8282352941176471,
  "acc_objetos_2cls": 0.8073394495412844,
  "f1_MILCO_objetos_2cls": 0.8571428571428571
 }
}


## B1 — backbones e resolução

In [3]:
A.tabela_classificadores(["B1"]).drop(columns=["grupo"])

,job,descrição,val AUC,teste acc,teste F1 macro,teste AUC,teste acc (TTA),épocas,"treino (min, CPU)",ms/recorte (CPU)
0,b1_convnextpico_128,ConvNeXt-Pico 128 px,0.923,0.908,0.893,0.962,0.927,29,6.6,7.5
1,b1_convnextpico_224,"ConvNeXt-Pico 224 px, fine-tuning completo, AdamW+cosine",0.946,0.899,0.879,0.932,0.908,22,16.0,24.3
2,b1_effv2b0_224,EfficientNetV2-B0 224 px,0.922,0.835,0.811,0.882,0.798,20,10.6,16.2
3,b1_mnv2_224,MobileNetV2 224 px (fine-tuning completo),0.881,0.780,0.751,0.849,0.780,20,12.6,14.9


**Leitura:** o ConvNeXt-Pico é o melhor backbone também nos recortes (acurácia de 0,90 a 0,91, contra 0,78 do MobileNetV2 nas mesmas condições). Em **128 px** ele é tão bom quanto em 224 px e custa **3× menos** (7,5 contra 24 ms por recorte em CPU).

## B2 — otimizadores e loss

In [4]:
A.tabela_classificadores(["B2"]).drop(columns=["grupo"])

,job,descrição,val AUC,teste acc,teste F1 macro,teste AUC,teste acc (TTA),épocas,"treino (min, CPU)",ms/recorte (CPU)
0,b2_adam_plateau,Adam + ReduceLROnPlateau (otimizador da fusão antiga/baseline),0.886,0.899,0.881,0.886,0.899,24,16.1,24.0
1,b2_focal,AdamW + Focal Loss (gamma 2),0.906,0.890,0.872,0.904,0.881,19,14.8,29.1
2,b2_label_smoothing,AdamW + label smoothing 0.1,0.941,0.844,0.814,0.907,0.862,25,18.9,30.4
3,b2_sam_adamw,SAM (rho 0.05) sobre AdamW,0.673,0.587,0.580,0.727,0.578,9,11.9,25.0
4,b2_sgd_lr002,SGD Nesterov lr 0.002 (backbone) ×10 na cabeça + cosine,0.523,0.688,0.408,0.438,0.688,9,6.7,31.9
5,b2_sgd_nesterov,SGD Nesterov lr 0.01 + cosine,0.406,0.688,0.408,0.520,0.688,9,6.0,27.2


**Leitura:**
- **AdamW + cosine continua o melhor.**
- **SAM e SGD colapsaram** neste problema pequeno: passaram a prever só MILCO (acurácia 0,69 = proporção de MILCO). É diferente da imagem inteira, onde o SAM empatou com o Adam.
- **Focal loss e Adam com LR constante** ficam próximos, sem ganho.
- **Label smoothing** tem boa AUC de validação, mas resultado de teste pior.

## B4 / B5 — fusões e o classificador de 3 classes

In [5]:
A.tabela_classificadores(["B4", "B5"]).drop(columns=["grupo"])

,job,descrição,val AUC,teste acc,teste F1 macro,teste AUC,teste acc (TTA),épocas,"treino (min, CPU)",ms/recorte (CPU)
0,b4_fusao_convnext_effv2,Fusão ConvNeXt-Pico + EfficientNetV2-B0,0.910,0.826,0.807,0.911,0.817,16,19.4,46.6
1,b4_fusao_convnext_ghostnet,Fusão ConvNeXt-Pico + GhostNet,0.899,0.807,0.770,0.886,0.826,16,16.9,35.4
2,b4_fusao_convnext_mnv2,Fusão ConvNeXt-Pico + MobileNetV2,0.914,0.761,0.707,0.815,0.761,30,35.7,42.4
3,b5_fundo_3classes,"3 classes (MILCO, NOMBO, fundo) para rejeitar falsos positivos do detector",0.909,0.927,0.923,0.951,0.927,29,39.3,27.6


**Leitura:**
- **Fusões não compensam:** dobram o custo e ficam abaixo do ConvNeXt sozinho, inclusive a fusão com GhostNet, análoga à antiga.
- **O classificador de 3 classes** teve a melhor acurácia nos objetos (0,927): aprender o que é "fundo" ajuda a separar MILCO de NOMBO.

## Finalistas com 3 sementes (média ± desvio-padrão)

In [6]:
A.tabela_sementes()

,config,val AUC,teste acc,teste AUC
0,b1_convnextpico_128,0.938 ± 0.011,0.911 ± 0.004,0.966 ± 0.005
1,b1_convnextpico_224,0.937 ± 0.007,0.887 ± 0.031,0.920 ± 0.033
2,b3_margem030,0.933 ± 0.003,0.755 ± 0.075,0.922 ± 0.013
3,b2_label_smoothing,0.917 ± 0.007,0.847 ± 0.026,0.891 ± 0.036
4,b5_fundo_3classes,0.909 ± 0.011,0.922 ± 0.026,0.933 ± 0.029


**Leitura:** com 3 sementes, os melhores (ConvNeXt-Pico 224, 128 e o de 3 classes) ficam entre **0,89 e 0,92 de acurácia**, contra **0,81 da fusão antiga**.
- **O 128 px é o mais estável** (desvio de 0,004).
- **Escolha do fluxo final:** não foi feita só pela AUC do recorte. Cada finalista foi colocado no fluxo completo (detector → recorte → classificador) e o escolhido foi o de **maior F1 de MILCO na validação**. Os detalhes estão no notebook 03.

In [7]:
# (Opcional) Re-treinar os experimentos deste notebook. Os resultados abaixo já vêm de resultados/runs/.
# Em máquina com GPU, troque RODAR_TREINO para True: o mesmo harness treina sem limite de tempo.
RODAR_TREINO = False
GRUPOS = ["B1", "B2", "B4", "B5", "F"]
import crop_harness as CH
fila = json.load(open("fila_recortes.json"))
jobs = [j for j in fila if j.get("group") in GRUPOS]
for j in jobs:
    print(f"{j['id']:32s} {j.get('desc','')}")
if RODAR_TREINO:
    import math
    for j in jobs:
        CH.train_job(j, deadline=math.inf)

b1_convnextpico_224              ConvNeXt-Pico 224 px, fine-tuning completo, AdamW+cosine
b1_convnextpico_128              ConvNeXt-Pico 128 px
b1_effv2b0_224                   EfficientNetV2-B0 224 px
b1_mnv2_224                      MobileNetV2 224 px (fine-tuning completo)
b2_adam_plateau                  Adam com LR constante (como a fusão antiga)
b2_sgd_nesterov                  SGD Nesterov lr 0.01 + cosine
b2_sam_adamw                     SAM (rho 0.05) sobre AdamW
b2_sgd_lr002                     SGD Nesterov lr 0.002 (backbone) ×10 na cabeça + cosine
b2_focal                         AdamW + Focal Loss (gamma 2)
b2_label_smoothing               AdamW + label smoothing 0.1
b4_fusao_convnext_mnv2           Fusão ConvNeXt-Pico + MobileNetV2
b4_fusao_convnext_effv2          Fusão ConvNeXt-Pico + EfficientNetV2-B0
b4_fusao_convnext_ghostnet       Fusão ConvNeXt-Pico + GhostNet
b5_fundo_3classes                3 classes (MILCO, NOMBO, fundo) para rejeitar falsos positivos do detector